# Week 4 · Quantum software stack and QCVV

**MIT xPRO QCFx1 · Week 4 · Videos Q1M4V4a and Q1M4V4b**

This notebook collects the software-stack and characterization material from V4.
It ends with a small *toy* noise experiment. That experiment is not randomized
benchmarking and not gate-set tomography; it only shows why real devices need
characterization.

Time reference: **Paramaribo (UTC−3)**. Hard deadline: **Mon 2026-11-02 20:30**.

## 1. The classical-to-quantum tool analogy

| Classical | Quantum |
|---|---|
| Schematic capture CAD | IBM Composer |
| Compiler (C++, VHDL/Verilog) | Quipper (Dalhousie), Q# → intermediate **QASM** |
| Assembler | Hardware mapping (Qiskit, superconducting): native gates + connectivity |
| SPICE (power, timing, correctness) | Quantum simulation: noise/error impact, gate fidelity under control waveforms |
| JTAG / boundary scan | **QCVV** (quantum characterization, verification, validation) |

Three programming steps:

1. **Program generation** — hardware-agnostic circuit or algorithm.
2. **Hardware-specific circuit mapping** — native gates, connectivity, routing.
3. **Hardware control and execution** — pulses, microwave sources, readout.

Three program-generation methods from the video:

- schematic capture (Composer, knows hardware constraints),
- high-level languages (Quipper, Q#, Scaffold),
- problem-specific generation (Google OpenFermion, quantum chemistry).

The key separation is that **control** is classical and room-temperature, while
**processing** is quantum and millikelvin. The two systems are often physically
apart; the dilution fridge is not sitting next to the laptop.

## 2. Control hardware and QCVV

Control hardware includes arbitrary waveform generators, microwave sources, and
lasers. Software includes Labber for commercial instruments, Qiskit for
superconducting devices, ARTIQ for ion traps, and many homegrown stacks.

Two characterization methods from the video:

| Method | What it gives | Cost / limitation |
|---|---|---|
| **Randomized benchmarking (RB)** | Average fidelity from long sequences of the target gate interleaved with random gates; mitigates state-prep and measurement (SPAM) error | One metric only; lots of data; small systems |
| **Gate-set tomography (GST)** | Full process maps (operator description of gate and error) | More complete but much more data; small systems |

Both need many measurements, so they are limited to small systems. Scalable QCVV
is open research. QuTiP (Python, open quantum systems) can model qubits coupled
to an environment, estimate coherence times and noise sensitivity, and simulate
logical error rates for quantum error correction. Simulation also helps size
"supremacy" circuits on HPC simulators.

In [ ]:
from pathlib import Path
import sys

def find_first_code_dir() -> Path:
    candidates = [Path.cwd(), Path.cwd() / "first-code", Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "firstcode_helpers.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repo root, first-code/, or first-code/notebooks/."
    )

FIRSTCODE_DIR = find_first_code_dir()
if str(FIRSTCODE_DIR) not in sys.path:
    sys.path.insert(0, str(FIRSTCODE_DIR))

print(f"First-code folder: {FIRSTCODE_DIR}")

import math
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit
import qiskit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_histogram
import firstcode_helpers as w4

print(f"Qiskit version: {qiskit.__version__}")

## 3. Ideal Bell state vs a toy noisy device

A Bell circuit is short enough that the ideal result is easy to predict: `00`
and `11` should each appear about half the time. A real device adds gate and
readout errors, so other bitstrings appear.

The noise model below is deliberately simple: depolarizing noise on every `H`
and `CX`. It is a teaching model, not a calibration of any IBM device.

In [ ]:
try:
    from qiskit_aer import AerSimulator
    from qiskit_aer.noise import NoiseModel, depolarizing_error

    bell = w4.bell_circuit(measure=True)
    ideal_counts = w4.run_counts(bell, shots=4096, seed=42)

    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 1), ["h"])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(0.02, 2), ["cx"])
    noisy_simulator = AerSimulator(noise_model=noise_model)
    noisy_counts = dict(
        noisy_simulator.run(
            bell, shots=4096, seed_simulator=42
        ).result().get_counts()
    )

    ideal_probabilities = w4.probabilities(ideal_counts, 4096)
    noisy_probabilities = w4.probabilities(noisy_counts, 4096)
    distance = w4.total_variation_distance(ideal_probabilities, noisy_probabilities)

    print("Ideal counts:", dict(sorted(ideal_counts.items())))
    print("Noisy counts:", dict(sorted(noisy_counts.items())))
    print(f"Total variation distance: {distance:.4f}")
    display(
        plot_histogram(
            [ideal_counts, noisy_counts],
            legend=["ideal", "toy noise"],
            title="Bell state: ideal vs depolarizing noise",
        )
    )
except ImportError:
    print("qiskit-aer is not installed; the ideal local cells still work.")

**Read this correctly:** the extra `01` and `10` outcomes are noise, not
entanglement. The total variation distance is a simple summary of how far the
noisy distribution is from the ideal one. It is not a gate fidelity, not an RB
number, and not a GST process map.

## 4. A toy sequence-survival experiment (not RB)

RB uses random gates and an inverse to isolate average gate error. The cell below
is much simpler: it applies $H$ twice (net identity) and lets depolarizing noise
accumulate with depth. The survival probability should fall as the sequence gets
longer. That is the intuition behind sequence fidelity, but the numbers are not
an RB decay curve.

In [ ]:
try:
    from qiskit_aer import AerSimulator
    from qiskit_aer.noise import NoiseModel, depolarizing_error

    toy_noise = NoiseModel()
    toy_noise.add_all_qubit_quantum_error(depolarizing_error(0.002, 1), ["h"])
    toy_simulator = AerSimulator(noise_model=toy_noise)

    print("depth  P(measure 0)  (ideal would be 1.0)")
    for depth in (1, 2, 4, 8, 16):
        circuit = QuantumCircuit(1)
        for _ in range(depth):
            circuit.h(0)
            circuit.h(0)
        circuit.measure_all()
        counts = toy_simulator.run(
            circuit, shots=2048, seed_simulator=42
        ).result().get_counts()
        survival = counts.get("0", 0) / 2048
        print(f"{depth:5d}  {survival:.4f}")
except ImportError:
    print("qiskit-aer is not installed; skipping the toy noise experiment.")

## 5. Why the distinction matters

- RB answers "what is the average fidelity of this gate set?" without needing a
  full model of the noise. It is efficient and SPAM-insensitive, but it collapses
  the error information into one number.
- GST answers "what process actually happens?" with a full operator description.
  It is more informative but expensive.
- Neither method scales cheaply to large devices. That is why scalable QCVV is
  an open research topic.

When you read a hardware calibration page, ask which question the number answers:
average fidelity, a specific process map, or a readout metric. They are not
interchangeable.

## 6. Predict before you run

1. Why does the Bell circuit produce only `00` and `11` in the ideal case?
2. Which noise channel would you add to model a readout error, as opposed to a
   gate error?
3. Why does the toy survival probability decrease with depth?
4. Why is the total variation distance not a substitute for RB?

Write your answer before running the noise cells. Then compare.